# KDE bandwidth playground and non-$L^2$ spectral scale selection

This notebook is an experimental environment for bandwidth selection in kernel density estimation. We sample

$$
X_1,\dots,X_n\sim \mathcal N(0,\sigma^2),
$$

and estimate the unknown density by the one-dimensional KDE

$$
\widehat{f}_h(x)=\frac{1}{nh}\sum_{r=1}^n K\left(\frac{x-X_r}{h}\right).
$$

The bandwidth $h>0$ is selected from a grid

$$
h_1<\dots<h_N.
$$

The notebook contains four types of experiments:

1. direct KDE visualization for chosen sample and bandwidth parameters;
2. an oracle bandwidth computed from the known density in simulation;
3. similarity matrices constructed from total variation, Hellinger affinity, and Jensen--Shannon divergence;
4. simulation studies of the resulting spectral bandwidth selectors.

The spectral selectors compute a nonnegative similarity matrix for the KDE family, take its principal eigenvector, and select the bandwidth at its largest component. The oracle is unavailable for real data and is used only as a benchmark.

> Pairwise beta-2/$L^2$ divergence experiments are maintained exclusively in [`spectral_scale_selection_kde.ipynb`](spectral_scale_selection_kde.ipynb).

The shared similarity selectors use the centralized symmetric Perron solver in `kde_tools.perron`.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib-cache"))

import numpy as np
import matplotlib.pyplot as plt                                                                     ### PLOTTING

from kde_tools.perron import PERRON_SOLVER_VERSION
from kde_tools.pdf_export import enable_automatic_pdf_saving                                        ### PLOTTING
from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta
from kde_tools.density import exact_oracle_ise_values, gaussian_kde_values
from kde_tools.experiments import run_one_consistency_experiment
from kde_tools.gaussian_mixtures import gaussian_density_with_variance
from kde_tools.plotting_functions import (                                                          ### PLOTTING
    plot_bandwidth_distributions,                                                                   ### PLOTTING
    plot_perron_vector,                                                                             ### PLOTTING
)                                                                                                   ### PLOTTING
from kde_tools.quadrature import (
    quadrature_grid_for_kdes,
    quadrature_similarity_matrices,
    select_quadrature_similarity_bandwidths,
    trapezoidal_integral,
)
from kde_tools.similarities import spectral_choice_from_similarity

pdf_output_directory = enable_automatic_pdf_saving(                                                 ### PLOTTING
    "100_kde_test"                                                                                  ### PLOTTING
)                                                                                                   ### PLOTTING

plt.style.use("seaborn-v0_8-whitegrid")

print(f"Perron solver: {PERRON_SOLVER_VERSION}")


## 1. Kernel functions and KDE implementation

The first code block defines the two kernels used throughout the notebook: Gaussian and Epanechnikov. Each kernel $K$ is normalized so that

$$

\int_{\mathbb R} K(u)\,du=1.

$$

For bandwidth $h$, the rescaled kernel is

$$

K_h(x)=\frac{1}{h}K\left(\frac{x}{h}\right),

$$

and therefore the KDE can also be written as

$$

\widehat{f}_h(x)=\frac{1}{n}\sum_{r=1}^n K_h(x-X_r).

$$

In the code, `kde_eval(x_grid, samples, h, kernel)` supports both kernels for the introductory plots. The divergence experiments below use the shared Gaussian-KDE and quadrature helpers from `kde_tools`.


In [ ]:
AVAILABLE_KERNELS = [
    "gaussian",
    "epanechnikov",
]


def kernel_values(u, kernel="gaussian"):
    """Evaluate a normalized one-dimensional Gaussian or Epanechnikov kernel K(u)."""
    u = np.asarray(u)
    kernel = kernel.lower()

    if kernel == "gaussian":
        return np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi)

    if kernel == "epanechnikov":
        out = np.zeros_like(u, dtype=float)
        inside = np.abs(u) <= 1
        out[inside] = 0.75 * (1 - u[inside] ** 2)
        return out

    raise ValueError(f"Unknown kernel: {kernel!r}. Choose one of {AVAILABLE_KERNELS}.")


def kde_eval(x_grid, samples, h, kernel="gaussian"):
    """Evaluate the KDE on x_grid for data samples and bandwidth h."""
    samples = np.asarray(samples)
    x_grid = np.asarray(x_grid)

    if h <= 0:
        raise ValueError("The bandwidth h must be positive.")
    if len(samples) == 0:
        raise ValueError("Need at least one sample point.")

    u = (x_grid[:, None] - samples[None, :]) / h
    return kernel_values(u, kernel=kernel).mean(axis=1) / h


## 2. Static KDE example

This block samples data once and computes one KDE for one manually chosen bandwidth. It is the simplest sanity check for the estimator.

The parameters are:

- `n`: number of sample points;
- `sigma2`: variance of the true normal distribution;
- `h`: KDE bandwidth;
- `kernel`: kernel shape;
- `seed`: random seed, so the experiment is reproducible.

The plot overlays the histogram, the KDE $\widehat{f}_h$, the true density $f$, and the sample locations. This lets you visually inspect whether the chosen bandwidth undersmooths or oversmooths.


In [ ]:
# Parameters you can change
n = 200
sigma2 = 1.0
h = 0.25
kernel = "gaussian"
seed = 7

rng = np.random.default_rng(seed)
X = rng.normal(loc=0.0, scale=np.sqrt(sigma2), size=n)

x_min = X.min() - 4 * h - 0.5
x_max = X.max() + 4 * h + 0.5
x_grid = np.linspace(x_min, x_max, 800)

kde = kde_eval(x_grid, X, h=h, kernel=kernel)
truth = gaussian_density_with_variance(x_grid, mean=0.0, variance=sigma2)

fig, ax = plt.subplots(figsize=(10, 5))                                                             ### PLOTTING
ax.hist(X, bins="auto", density=True, alpha=0.25, color="gray", label="sample histogram")           ### PLOTTING
ax.plot(x_grid, kde, lw=2.5, label=f"KDE, kernel={kernel}, h={h}")                                  ### PLOTTING
ax.plot(x_grid, truth, "--", lw=2, label=fr"true $N(0,{sigma2})$ density")                          ### PLOTTING
ax.scatter(X, np.zeros_like(X), marker="|", s=80, color="black", alpha=0.45, label="sample points") ### PLOTTING
ax.set_title("Kernel density estimate")
ax.set_xlabel("x")                                                                                  ### PLOTTING
ax.set_ylabel("density")                                                                            ### PLOTTING
ax.legend()                                                                                         ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 3. Compare several bandwidths

Here we hold the data set fixed and evaluate several KDEs

$$

\widehat{f}_{h_1},\widehat{f}_{h_2},\dots

$$

on the same grid. This isolates the effect of the bandwidth from the randomness of the sample.

The qualitative behavior should be:

- small $h$: many narrow peaks, high variance, low bias;
- large $h$: one broad smooth curve, low variance, high bias;
- intermediate $h$: often the best compromise.

This is the visual intuition behind bandwidth selection.


In [ ]:
n = 200
sigma2 = 1.0
kernel = "gaussian"
bandwidths = [0.08, 0.15, 0.30, 0.60, 1.20]
seed = 7

rng = np.random.default_rng(seed)
X = rng.normal(loc=0.0, scale=np.sqrt(sigma2), size=n)
x_grid = np.linspace(X.min() - 2.5, X.max() + 2.5, 1000)

fig, ax = plt.subplots(figsize=(10, 5))                                                             ### PLOTTING
ax.hist(X, bins="auto", density=True, alpha=0.18, color="gray", label="sample histogram")           ### PLOTTING
ax.plot(x_grid, gaussian_density_with_variance(x_grid, mean=0.0, variance=sigma2), "k--", lw=2, label="true density")  ### PLOTTING

for h_value in bandwidths:
    kde = kde_eval(x_grid, X, h=h_value, kernel=kernel)
    ax.plot(x_grid, kde, lw=2, label=f"h={h_value:g}")                                              ### PLOTTING

ax.set_title(f"Bandwidth comparison using the {kernel} kernel")
ax.set_xlabel("x")                                                                                  ### PLOTTING
ax.set_ylabel("density")                                                                            ### PLOTTING
ax.legend(ncol=2)                                                                                   ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 4. Interactive KDE plot

This section repeats the static KDE experiment with sliders. The most important control is the bandwidth slider $h$. Moving it shows the scale-space behavior of the KDE in real time.

Mathematically, changing $h$ changes the amount of smoothing in

$$

\widehat{f}_h(x)=\frac{1}{nh}\sum_{r=1}^nK\left(\frac{x-X_r}{h}\right).

$$

The sample itself changes only when `seed`, `n`, or `sigma²` changes. If `ipywidgets` is unavailable, the rest of the notebook still works; only this interactive control block is skipped.


In [ ]:
try:
    from ipywidgets import Button, Dropdown, FloatLogSlider, HBox, IntSlider, Output, VBox
    from IPython.display import display                                                             ### REPORTING
    widgets_available = True
except ModuleNotFoundError:
    widgets_available = False
    print(                                                                                          ### REPORTING
        "ipywidgets is not installed in this Python environment. "                                  ### REPORTING
        "Install it with `pip install ipywidgets` or `conda install ipywidgets` "                   ### REPORTING
        "to use the interactive sliders. The non-interactive KDE and matrix cells still work."      ### REPORTING
    )                                                                                               ### REPORTING


if widgets_available:
    n_slider = IntSlider(value=200, min=20, max=2000, step=20, description="n")
    sigma2_slider = FloatLogSlider(
        value=1.0,
        base=10,
        min=-2,
        max=1.3,
        step=0.05,
        description="sigma²",
        readout_format=".3g",
    )
    seed_slider = IntSlider(value=7, min=0, max=999, step=1, description="seed")
    kernel_dropdown = Dropdown(options=AVAILABLE_KERNELS, value="gaussian", description="kernel")
    h_slider = FloatLogSlider(
        value=0.25,
        base=10,
        min=-2,
        max=1.1,
        step=0.01,
        description="h",
        readout_format=".3g",
        continuous_update=True,
    )
    resample_button = Button(description="resample", button_style="")
    out = Output()

    def draw_interactive_kde(*_):
        n = n_slider.value
        sigma2 = sigma2_slider.value
        seed = seed_slider.value
        h = h_slider.value
        kernel = kernel_dropdown.value

        rng = np.random.default_rng(seed)
        X = rng.normal(loc=0.0, scale=np.sqrt(sigma2), size=n)

        # Use a plotting window that adapts to both the data spread and the current bandwidth.
        margin = max(4 * np.sqrt(sigma2), 6 * h, 1.0)
        x_grid = np.linspace(X.min() - margin, X.max() + margin, 1000)
        kde = kde_eval(x_grid, X, h=h, kernel=kernel)
        truth = gaussian_density_with_variance(x_grid, mean=0.0, variance=sigma2)

        with out:
            out.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 5.5))                                               ### PLOTTING
            ax.hist(X, bins="auto", density=True, alpha=0.20, color="gray", label="sample histogram")  ### PLOTTING
            ax.plot(x_grid, kde, lw=2.5, color="tab:blue", label=f"KDE, h={h:.3g}")                 ### PLOTTING
            ax.plot(x_grid, truth, "--", lw=2, color="black", label=fr"true $N(0,{sigma2:.3g})$ density")  ### PLOTTING
            ax.scatter(X, np.zeros_like(X), marker="|", s=65, color="black", alpha=0.35)            ### PLOTTING
            ax.set_title(f"Interactive KDE — {kernel} kernel, n={n}, sigma²={sigma2:.3g}")
            ax.set_xlabel("x")                                                                      ### PLOTTING
            ax.set_ylabel("density")                                                                ### PLOTTING
            ax.legend(loc="upper right")                                                            ### PLOTTING
            plt.show()                                                                              ### PLOTTING

    def advance_seed(_):
        seed_slider.value = seed_slider.value + 1 if seed_slider.value < seed_slider.max else seed_slider.min

    for widget in [n_slider, sigma2_slider, seed_slider, kernel_dropdown, h_slider]:
        widget.observe(draw_interactive_kde, names="value")

    resample_button.on_click(advance_seed)

    top_controls = HBox([n_slider, sigma2_slider])
    middle_controls = HBox([kernel_dropdown, seed_slider, resample_button])

    display(VBox([top_controls, middle_controls, out, h_slider]))                                   ### REPORTING
    draw_interactive_kde()


## 5. Shared simulation setup and numerical helper formulas

From here on the notebook uses one common simulated sample

$$
X_1,\dots,X_n\sim N(0,\sigma^2),
$$

a bandwidth grid

$$
h_1<\cdots<h_N,
$$

and the corresponding KDE family

$$
\widehat f_{h_1},\dots,\widehat f_{h_N}.
$$

The densities and integrated squared errors are evaluated on a common numerical integration grid. Integrated squared error is retained only as an oracle evaluation benchmark; it is not used here to build a pairwise spectral similarity.


In [ ]:
# Shared state used by the divergence-based scale-selection cells below.
n_scale = 200
sigma2 = 1.0
seed_scale = 123
n_h = 80

rng = np.random.default_rng(seed_scale)
X = rng.normal(loc=0.0, scale=np.sqrt(sigma2), size=n_scale)

# The grid is centered around the usual n^{-1/5} bandwidth scale, but spans
# substantially smaller and larger bandwidths.
h_center = np.sqrt(sigma2) * n_scale ** (-1 / 5)
h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=0.20 * h_center,
    h_max=3.00 * h_center,
    beta=2.0,
    grid_size=n_h,
)

x_grid = quadrature_grid_for_kdes(
    samples=X,
    h_grid=h_grid,
    m_x=2_001,
)
kde_family = gaussian_kde_values(
    samples=X,
    h_grid=h_grid,
    x_grid=x_grid,
)
truth = gaussian_density_with_variance(
    x=x_grid,
    mean=0.0,
    variance=sigma2,
)

# Exact Gaussian-KDE ISE benchmark against the known N(0, sigma^2) density.
ise_values = exact_oracle_ise_values(
    samples=X,
    h_grid=h_grid,
    sigma2=sigma2,
)
oracle_index = int(np.argmin(ise_values))
oracle_h = float(h_grid[oracle_index])

print("Shared KDE state rebuilt")                                                                   ### REPORTING
print(f"sample size n = {n_scale}")                                                                 ### REPORTING
print("kernel = gaussian")                                                                          ### REPORTING
print(f"oracle h = {oracle_h:.6g}")                                                                 ### REPORTING


## 6. Oracle bandwidth from the known true density

Because the data in this notebook are simulated from a known density

$$
f(x)=N(0,\sigma^2),
$$

we can compute an oracle benchmark. For every candidate bandwidth, we evaluate

$$
ISE(h)=\int(\widehat f_h(x)-f(x))^2\,dx.
$$

The oracle bandwidth is

$$
h_{oracle}=\arg\min_{h_i} ISE(h_i).
$$

This is not available for real data; it is only a benchmark for the other selectors.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))                                                             ### PLOTTING
ax.plot(h_grid, ise_values, marker="o", ms=4, lw=2, label="ISE curve")                              ### PLOTTING
ax.axvline(oracle_h, color="tab:green", ls="-", lw=2, label=f"oracle h={oracle_h:.3g}")             ### PLOTTING
ax.set_xscale("log")                                                                                ### PLOTTING
ax.set_title("Oracle bandwidth via integrated squared error")
ax.set_xlabel("bandwidth h")                                                                        ### PLOTTING
ax.set_ylabel("ISE(h)")                                                                             ### PLOTTING
ax.legend()                                                                                         ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 7. Non-$L^2$ similarity matrices

This section constructs three pairwise similarities for the KDE family. TV overlap, Hellinger affinity, and JS use composite trapezoidal integration on the shared spatial grid. Increase the spatial resolution and interval padding to check convergence. The implementation does not provide a total numerical error certificate. See [the shared quadrature implementation](kde_tools/quadrature.py).

The Jensen--Shannon construction uses

$$
S^{JS}_{ij}=1-\frac{JS(\widehat f_i,\widehat f_j)}{\log 2}.
$$

The Hellinger affinity and total-variation overlap are

$$
S^H_{ij}=\int \sqrt{\widehat f_i(x)\widehat f_j(x)}\,dx,
\qquad
S^{TV}_{ij}=\int \min\{\widehat f_i(x),\widehat f_j(x)\}\,dx.
$$

Each nonnegative similarity matrix is summarized by its principal eigenvector, and the bandwidth at its largest component is selected. The beta-2/$L^2$ construction is intentionally omitted because it belongs to the main $L^2$ notebook.


In [ ]:
# Normalize the numerically evaluated KDEs before constructing similarities.
kde_family_normalized = kde_family / trapezoidal_integral(
    kde_family,
    x_grid=x_grid,
    axis=1,
)[:, None]

similarity_matrices = {
    name: quadrature_similarity_matrices(
        density_values=kde_family_normalized,
        x_grid=x_grid,
        similarity_name=name,
    )
    for name in (
        "tv_overlap",
        "hellinger_affinity",
        "js_similarity",
    )
}

TV_overlap = similarity_matrices["tv_overlap"]
Hellinger_affinity = similarity_matrices["hellinger_affinity"]
JS_similarity = similarity_matrices["js_similarity"]

choices = {
    name: spectral_choice_from_similarity(matrix)
    for name, matrix in similarity_matrices.items()
}

perron_tv = choices["tv_overlap"]["perron_vector"]
perron_hellinger = choices["hellinger_affinity"]["perron_vector"]
perron_js = choices["js_similarity"]["perron_vector"]

selected_index_tv = int(choices["tv_overlap"]["selected_index"])
selected_index_hellinger = int(choices["hellinger_affinity"]["selected_index"])
selected_index_js = int(choices["js_similarity"]["selected_index"])

selected_h_tv = float(h_grid[selected_index_tv])
selected_h_hellinger = float(h_grid[selected_index_hellinger])
selected_h_js = float(h_grid[selected_index_js])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)                                      ### PLOTTING
for ax, name, vec, h_sel in [
    (axes[0], "TV overlap", perron_tv, selected_h_tv),
    (axes[1], "Hellinger affinity", perron_hellinger, selected_h_hellinger),
    (axes[2], "JS similarity", perron_js, selected_h_js),
]:
    plot_perron_vector(h_grid=h_grid, vector=vec, ax=ax)                                            ### PLOTTING
    ax.set_title(f"{name}\nh={h_sel:.3g}")
plt.tight_layout()                                                                                  ### PLOTTING
plt.show()                                                                                          ### PLOTTING

print(f"JS selected h = {selected_h_js:.6g}")                                                       ### REPORTING
print(f"Hellinger selected h = {selected_h_hellinger:.6g}")                                         ### REPORTING


## 8. Compare the selected bandwidths

This plot compares the non-$L^2$ spectral selectors with the oracle benchmark.


In [ ]:
selection_summary = [
    ("Oracle ISE", oracle_h, oracle_index),
    ("TV overlap", selected_h_tv, selected_index_tv),
    ("Hellinger", selected_h_hellinger, selected_index_hellinger),
    ("JS", selected_h_js, selected_index_js),
]

fig, ax = plt.subplots(figsize=(11, 5))                                                             ### PLOTTING
for k, (name, h_sel, _) in enumerate(selection_summary):                                            ### PLOTTING
    ax.scatter(h_sel, k, s=90)                                                                      ### PLOTTING
    ax.text(h_sel, k + 0.08, f"{h_sel:.3g}", ha="center", fontsize=9)                               ### PLOTTING
ax.set_xscale("log")                                                                                ### PLOTTING
ax.set_yticks(range(len(selection_summary)))                                                        ### PLOTTING
ax.set_yticklabels([name for name, _, _ in selection_summary])                                      ### PLOTTING
ax.set_xlabel("selected bandwidth h")                                                               ### PLOTTING
ax.set_title("Bandwidths selected by non-$L^2$ similarity methods")
plt.show()                                                                                          ### PLOTTING

for name, h_sel, idx in selection_summary:                                                          ### REPORTING
    print(f"{name:18s} index={idx:3d}, h={h_sel:.6g}, ISE={ise_values[idx]:.6g}")                   ### REPORTING


## 9. Inspect the KDE selected by the entropy/JS matrix

This plot shows the KDE selected by the entropy/Jensen-Shannon matrix together with the true Gaussian density. The histogram, oracle KDE, and rug marks are intentionally omitted so the comparison stays visually focused: selected KDE versus known ground truth.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)                                      ### PLOTTING
for ax, name, idx in [
    (axes[0], "JS similarity", selected_index_js),
    (axes[1], "Hellinger affinity", selected_index_hellinger),
]:
    ax.hist(X, bins="auto", density=True, alpha=0.22, color="gray")                                 ### PLOTTING
    ax.plot(x_grid, truth, "--", lw=2, color="black", label="true density")                         ### PLOTTING
    ax.plot(x_grid, kde_family[idx], lw=2.5, label=fr"KDE, h={h_grid[idx]:.3g}")                    ### PLOTTING
    ax.scatter(X, np.zeros_like(X), marker="|", s=55, color="black", alpha=0.30)                    ### PLOTTING
    ax.set_title(name)
    ax.set_xlabel("x")                                                                              ### PLOTTING
    ax.legend()                                                                                     ### PLOTTING
axes[0].set_ylabel("density")                                                                       ### PLOTTING
plt.tight_layout()                                                                                  ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 10. Fixed-$n$ simulation: empirical bandwidth distributions for oracle and JS quadrature

In this experiment the sample size is fixed. We repeat the following procedure $R$ times (currently $R=30$):

1. draw a new sample

$$
X_1,\dots,X_n\sim N(0,\sigma^2),
$$

2. compute the oracle bandwidth

$$
h_{oracle}=\arg\min_h \int (\widehat f_h(x)-f(x))^2\,dx,
$$

3. compute the honest Jensen-Shannon bandwidth using composite trapezoidal integration,

$$
h_{JS}^{grid}=h_{\arg\max_i v_i},
$$

where $v$ is the Perron vector of the JS similarity matrix computed by composite trapezoidal integration on the shared spatial grid.

The plot shows the empirical relative frequencies with which each bandwidth grid value is selected by the two methods.


In [ ]:
num_simulations_distribution = 30
n_distribution = 200
sigma2_distribution = 1.0
n_h_distribution = 60
base_seed_distribution = 91_000

h_center_distribution = (
    np.sqrt(sigma2_distribution) * n_distribution ** (-1 / 5)
)
h_grid_distribution = bandwidth_grid_linear_in_psi_beta(
    h_min=0.20 * h_center_distribution,
    h_max=3.00 * h_center_distribution,
    beta=2.0,
    grid_size=n_h_distribution,
)

results_distribution = []
for simulation in range(num_simulations_distribution):
    rows = run_one_consistency_experiment(
        seed=base_seed_distribution + simulation,
        n=n_distribution,
        sigma2=sigma2_distribution,
        h_grid=h_grid_distribution,
        m_x=1_501,
    )
    index_by_method = {
        row["method"]: int(row["selected_index"])
        for row in rows
    }
    results_distribution.append(
        (
            index_by_method["Oracle ISE"],
            index_by_method["JS similarity"],
            index_by_method["Hellinger affinity"],
        )
    )

results_distribution = np.asarray(results_distribution, dtype=int)
labels_and_columns = [
    ("Oracle ISE", 0),
    ("JS similarity", 1),
    ("Hellinger affinity", 2),
]
counts_by_method = {
    label: np.bincount(
        results_distribution[:, column],
        minlength=len(h_grid_distribution),
    )
    for label, column in labels_and_columns
}

figure = plot_bandwidth_distributions(                                                              ### PLOTTING
    h_grid=h_grid_distribution,                                                                     ### PLOTTING
    distributions=counts_by_method,                                                                 ### PLOTTING
)                                                                                                   ### PLOTTING
figure.show()                                                                                       ### PLOTTING

for label, column in labels_and_columns:
    indices = results_distribution[:, column]
    modal_index = int(np.argmax(counts_by_method[label]))
    print(                                                                                          ### REPORTING
        f"{label:20s}: modal h = {h_grid_distribution[modal_index]:.6g}, "                          ### REPORTING
        f"median h = {np.median(h_grid_distribution[indices]):.6g}"                                 ### REPORTING
    )                                                                                               ### REPORTING


## 11. Consistency-style plot: sample size versus selected KDE error

This experiment checks the direct consistency quantity

$$
\left\|f-\widehat f_{\widehat h_n}\right\|_2.
$$

For each sample size $n$, we repeat the simulation several times. In each repetition we choose two bandwidths:

1. the oracle bandwidth from minimum integrated squared error;
2. the JS-quadrature bandwidth from the Perron vector of the JS matrix computed by composite trapezoidal integration.

Because the true density is known in this simulation, we can compute the exact integrated squared error at both selected bandwidths. The plotted quantity is

$$
\left\|f-\widehat f_{\widehat h_n}\right\|_2
=
\sqrt{ISE(\widehat h_n)}.
$$

If the selected KDE is consistent, this error should decrease as $n$ grows.


In [ ]:
n_values_consistency = np.arange(100, 1001, 100)
num_simulations_consistency = 10
n_h_consistency = 50
base_seed_consistency = 92_000
sigma2_consistency = 1.0

consistency_rows = []
for n_current in n_values_consistency:
    h_center_current = (
        np.sqrt(sigma2_consistency) * int(n_current) ** (-1 / 5)
    )
    h_grid_current = bandwidth_grid_linear_in_psi_beta(
        h_min=0.20 * h_center_current,
        h_max=3.00 * h_center_current,
        beta=2.0,
        grid_size=n_h_consistency,
    )
    for simulation in range(num_simulations_consistency):
        consistency_rows.extend(
            run_one_consistency_experiment(
                seed=(
                    base_seed_consistency
                    + 10_000 * int(n_current)
                    + simulation
                ),
                n=int(n_current),
                sigma2=sigma2_consistency,
                h_grid=h_grid_current,
                m_x=1_501,
            )
        )

fig, ax = plt.subplots(figsize=(10, 5))                                                             ### PLOTTING
for method in ["Oracle ISE", "JS similarity", "Hellinger affinity"]:
    medians = []
    q25 = []
    q75 = []
    for n_current in n_values_consistency:
        values = np.asarray([
            row["squared_integrated_error"]
            for row in consistency_rows
            if row["n"] == int(n_current) and row["method"] == method
        ])
        medians.append(np.median(values))
        q25.append(np.quantile(values, 0.25))
        q75.append(np.quantile(values, 0.75))
    medians = np.asarray(medians)
    q25 = np.asarray(q25)
    q75 = np.asarray(q75)
    ax.plot(n_values_consistency, medians, marker="o", lw=2.2, label=method)                        ### PLOTTING
    ax.fill_between(n_values_consistency, q25, q75, alpha=0.18)                                     ### PLOTTING
ax.set_yscale("log")                                                                                ### PLOTTING
ax.set_title("Sample size versus selected KDE squared integrated error")
ax.set_xlabel("sample size n")                                                                      ### PLOTTING
ax.set_ylabel(r"$\int (\widehat f_{\widehat h}-f)^2$")                                              ### PLOTTING
ax.legend()                                                                                         ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 12. Rate-controlled Hellinger selector: forcing the correct KDE scaling

The spectral selector only chooses from the bandwidth grid that we give it. Therefore, to make the asymptotic scaling explicit, we use a grid of the form

$$
h_i^{(n)}=c_i\, n^{-1/5},
\qquad
0<c_1<\cdots<c_N.
$$

The constants $c_i$ are fixed across sample sizes. Hence every bandwidth selected from this grid satisfies

$$
\widehat h_n\to0,
\qquad
n\widehat h_n\to\infty,
$$

as long as the selected constants stay inside the fixed finite interval $[c_1,c_N]$. This block tests the Hellinger spectral selector again under this simple rate-controlled grid with no unknown scale parameter.


In [ ]:
# Rate-controlled Hellinger experiment.
# The grid has the form h_i^(n) = c_i * n^(-1/5).
# Increase num_simulations_rate_controlled for final figures.
n_values_rate_controlled = np.arange(100, 1001, 100)
num_simulations_rate_controlled = 10
n_h_rate_controlled = 45
m_x_rate_controlled = 1_001
sigma2_rate_controlled = 1.0
seed_rate_controlled = 610_000
c_grid_rate_controlled = bandwidth_grid_linear_in_psi_beta(
    h_min=0.01,
    h_max=2.00,
    beta=2.0,
    grid_size=n_h_rate_controlled,
)


def select_hellinger_on_rate_controlled_grid(
    samples,
    n_current,
    c_grid,
    sigma2=1.0,
    m_x=1_001,
):
    """Oracle and Hellinger spectral selections on h_i = c_i n^{-1/5}."""
    h_grid_local = c_grid * n_current ** (-1 / 5)
    ise_values_local = exact_oracle_ise_values(
        samples=samples,
        h_grid=h_grid_local,
        sigma2=sigma2,
    )
    oracle_index_local = int(np.argmin(ise_values_local))

    selection = select_quadrature_similarity_bandwidths(
        samples=samples,
        h_grid=h_grid_local,
        similarity_name="hellinger_affinity",
        m_x=m_x,
    )
    choice = selection["choice"]
    hellinger_index_local = int(choice["selected_index"])

    return {
        "h_grid": h_grid_local,
        "oracle_index": oracle_index_local,
        "oracle_h": float(h_grid_local[oracle_index_local]),
        "oracle_c": float(c_grid[oracle_index_local]),
        "oracle_error": float(ise_values_local[oracle_index_local]),
        "hellinger_index": hellinger_index_local,
        "hellinger_h": float(h_grid_local[hellinger_index_local]),
        "hellinger_c": float(c_grid[hellinger_index_local]),
        "hellinger_error": float(ise_values_local[hellinger_index_local]),
        "hellinger_vector": choice["perron_vector"],
    }


rate_controlled_rows = []

for n_current in n_values_rate_controlled:
    for simulation in range(num_simulations_rate_controlled):
        rng = np.random.default_rng(seed_rate_controlled + 10_000 * int(n_current) + simulation)
        samples = rng.normal(0.0, np.sqrt(sigma2_rate_controlled), size=int(n_current))
        result = select_hellinger_on_rate_controlled_grid(
            samples=samples,
            n_current=int(n_current),
            c_grid=c_grid_rate_controlled,
            sigma2=sigma2_rate_controlled,
            m_x=m_x_rate_controlled,
        )
        rate_controlled_rows.append({
            "n": int(n_current),
            "simulation": int(simulation),
            "method": "Oracle ISE",
            "index": result["oracle_index"],
            "h": result["oracle_h"],
            "c": result["oracle_c"],
            "nh": int(n_current) * result["oracle_h"],
            "squared_integrated_error": result["oracle_error"],
        })
        rate_controlled_rows.append({
            "n": int(n_current),
            "simulation": int(simulation),
            "method": "Hellinger spectral",
            "index": result["hellinger_index"],
            "h": result["hellinger_h"],
            "c": result["hellinger_c"],
            "nh": int(n_current) * result["hellinger_h"],
            "squared_integrated_error": result["hellinger_error"],
        })

rate_controlled_results = np.array(rate_controlled_rows, dtype=object)

def rate_controlled_series(method, key, reducer=np.median):
    values = []
    for n_current in n_values_rate_controlled:
        current = [row[key] for row in rate_controlled_rows if row["n"] == int(n_current) and row["method"] == method]
        values.append(reducer(np.asarray(current, dtype=float)))
    return np.asarray(values, dtype=float)


fig, axes = plt.subplots(2, 2, figsize=(13, 9))                                                     ### PLOTTING
axes = axes.ravel()

for method in ["Oracle ISE", "Hellinger spectral"]:
    h_median = rate_controlled_series(method, "h")
    c_median = rate_controlled_series(method, "c")
    nh_median = rate_controlled_series(method, "nh")
    error_median = rate_controlled_series(method, "squared_integrated_error")

    axes[0].plot(n_values_rate_controlled, h_median, marker="o", lw=2.2, label=method)              ### PLOTTING
    axes[1].plot(n_values_rate_controlled, c_median, marker="o", lw=2.2, label=method)              ### PLOTTING
    axes[2].plot(n_values_rate_controlled, nh_median, marker="o", lw=2.2, label=method)             ### PLOTTING
    axes[3].plot(n_values_rate_controlled, error_median, marker="o", lw=2.2, label=method)          ### PLOTTING

guide = c_grid_rate_controlled[len(c_grid_rate_controlled) // 2] * n_values_rate_controlled ** (-1 / 5)
axes[0].plot(n_values_rate_controlled, guide, "k--", alpha=0.45, label=r"guide $n^{-1/5}$")         ### PLOTTING

axes[0].set_title(r"Selected bandwidth $\widehat h_n$")
axes[0].set_xlabel("sample size n")                                                                 ### PLOTTING
axes[0].set_ylabel("median selected h")                                                             ### PLOTTING
axes[0].set_yscale("log")                                                                           ### PLOTTING

axes[1].set_title(r"Selected constant $n^{1/5}\widehat h_n$")
axes[1].set_xlabel("sample size n")                                                                 ### PLOTTING
axes[1].set_ylabel("median selected c")                                                             ### PLOTTING

axes[2].set_title(r"Effective local sample size $n\widehat h_n$")
axes[2].set_xlabel("sample size n")                                                                 ### PLOTTING
axes[2].set_ylabel("median n h")                                                                    ### PLOTTING

axes[3].set_title("Selected KDE squared integrated error")
axes[3].set_xlabel("sample size n")                                                                 ### PLOTTING
axes[3].set_ylabel(r"median $\int(\widehat f_{\widehat h}-f)^2$")                                   ### PLOTTING
axes[3].set_yscale("log")                                                                           ### PLOTTING

for ax in axes:                                                                                     ### PLOTTING
    ax.legend()                                                                                     ### PLOTTING

plt.tight_layout()                                                                                  ### PLOTTING
plt.show()                                                                                          ### PLOTTING

print("Rate-controlled Hellinger grid")                                                             ### REPORTING
print("-------------------------------")                                                            ### REPORTING
print(r"Grid: h_i^(n) = c_i * n^(-1/5)")                                                            ### REPORTING
print(f"c_min = {c_grid_rate_controlled[0]:.6g}, c_max = {c_grid_rate_controlled[-1]:.6g}")         ### REPORTING
print("Therefore any selected bandwidth from this grid satisfies h_n -> 0 and n h_n -> infinity.")  ### REPORTING

